# ARTS : 

In [1]:
import pandas as pd 
df = pd.read_csv("/home/ppoulenard/LLM_BIAS_UCHILE/DATA/cultural_wiki_mcq_pt_clean.csv")


In [2]:
import unicodedata
import re

def normalize(title):
    # 1. Convertir en minuscules
    title = title.lower()

    # 2. Remplacer les espaces et underscores par un seul underscore
    title = re.sub(r'[\s_]+', '_', title)

    # 3. Supprimer les accents et caractères spéciaux
    title = unicodedata.normalize('NFKD', title)
    title = ''.join(c for c in title if not unicodedata.combining(c))

    # 4. Supprimer les parenthèses et leur contenu (ex: "Pizza (plat)" → "pizza")
    title = re.sub(r'\(.*\)', '', title)

    # 5. Supprimer les préfixes comme "categoría:" ou "catégorie:"
    title = re.sub(r'^categor[aí]a_', '', title, flags=re.IGNORECASE)

    # 6. Supprimer les espaces vides résiduels
    title = title.strip('_')

    return title

In [15]:
import requests
import time

HEADERS = {
    "User-Agent": "MyBot/1.0 (https://example.com/bot-info)"
}

def get_category_members(category, depth=2, lang="pt", visited=None):
    if visited is None:
        visited = set()
    if category in visited or depth < 0:
        return set()
    visited.add(category)

    api = f"https://{lang}.wikipedia.org/w/api.php"
    titles = set()
    subcats = []
    cmcontinue = None

    while True:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": category,
            "cmlimit": "500",
            "cmtype": "page|subcat",
            "format": "json",
            "formatversion": "2",
        }
        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        resp = requests.get(api, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()                # ← te montrera la vraie erreur HTTP
        try:
            r = resp.json()
        except Exception:
            print("Réponse non-JSON reçue :")
            print(resp.text[:500])             # debug
            raise

        for m in r.get("query", {}).get("categorymembers", []):
            if m["ns"] == 14:
                subcats.append(m["title"])
            elif m["ns"] == 0:
                titles.add(m["title"])

        if "continue" in r:
            cmcontinue = r["continue"]["cmcontinue"]
        else:
            break
        time.sleep(0.01)

    for sub in subcats:
        titles |= get_category_members(sub, depth=depth-1, lang=lang, visited=visited)
        time.sleep(0.01)

    return titles




categories_arts = [
    "Categoria:Literatura do Brasil"
]

all_titles = {}
for cat in categories_arts:
    print(f"→ {cat}")
    all_titles[cat] = get_category_members(cat, depth=2)
    print(f"   {len(all_titles[cat])} articles")


→ Categoria:Literatura do Brasil
   2785 articles


In [16]:
# Union globale
union_titles = set().union(*all_titles.values())
union_norm = {normalize(t) for t in union_titles}

import pandas as pd 
df = pd.read_csv("/home/ppoulenard/LLM_BIAS_UCHILE/DATA/cultural_wiki_mcq_pt_clean.csv")
print(df["title"])
df["title_norm"] = df["title"].astype(str).apply(normalize)


print(union_titles)
print(union_norm)
df_corpus = df[df["title_norm"].isin(union_norm)].copy()
print(f"\n✅ Corpus final : {len(df_corpus)} articles")
df_corpus.to_csv("/home/ppoulenard/LLM_BIAS_UCHILE/DATA/SUBSETS/PT/literatura_articles_pt.csv", index=False)

0                        Multiculturalismo no Brasil
1                                  José Lins do Rego
2                                       Miolo do boi
3                                     Samba de bumbo
4                                     Alfredo Wagner
                            ...                     
6072                           Nova matriz econômica
6073                             Grafismos indígenas
6074    Estação Ciência da Universidade de São Paulo
6075                                 Chapa de frente
6076                      Centro Histórico de Cuiabá
Name: title, Length: 6077, dtype: str
{'Adrianna Alberti', 'O Diletante', 'Tudo é Rio', 'As Américas e a Civilização', 'Nova Minigramática da Língua Portuguesa', 'Joca Reiners Terron', 'Pedagogia da Autonomia', 'Estrada Nova (livro)', 'Felisbelo da Silva', 'Adriana Lunardi', 'Hortênsia (romance)', 'Segunda Classe', 'Montenegro (livro)', 'Tudo por um Popstar', 'Prêmio ABL de Tradução', 'Maria Lúcia Medeiros', 'Suor

In [55]:
import requests
import time

HEADERS = {
    "User-Agent": "MyBot/1.0 (https://example.com/bot-info)"
}

def get_category_members(category, depth=2, lang="es", visited=None):
    if visited is None:
        visited = set()
    if category in visited or depth < 0:
        return set()
    visited.add(category)

    api = f"https://{lang}.wikipedia.org/w/api.php"
    titles = set()
    subcats = []
    cmcontinue = None

    while True:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": category,
            "cmlimit": "500",
            "cmtype": "page|subcat",
            "format": "json",
            "formatversion": "2",
        }
        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        resp = requests.get(api, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()                # ← te montrera la vraie erreur HTTP
        try:
            r = resp.json()
        except Exception:
            print("Réponse non-JSON reçue :")
            print(resp.text[:500])             # debug
            raise

        for m in r.get("query", {}).get("categorymembers", []):
            if m["ns"] == 14:
                subcats.append(m["title"])
            elif m["ns"] == 0:
                titles.add(m["title"])

        if "continue" in r:
            cmcontinue = r["continue"]["cmcontinue"]
        else:
            break
        time.sleep(0.01)

    for sub in subcats:
        titles |= get_category_members(sub, depth=depth-1, lang=lang, visited=visited)
        time.sleep(0.01)

    return titles




categories_arts = [
    "Categoría:Deporte en Argentina",
    "Categoría:Deporte en Bolivia",
    "Categoría:Deporte en Chile",
    "Categoría:Deporte en Colombia",
    "Categoría:Deporte en Ecuador",
    "Categoría:Deporte en México",
    "Categoría:Deporte en Paraguay",
    "Categoría:Deporte en Perú",
    "Categoría:Deporte en Uruguay",
    "Categoría:Deporte en Venezuela"
    # Variantes :
]

all_titles = {}
for cat in categories_arts:
    print(f"→ {cat}")
    all_titles[cat] = get_category_members(cat, depth=2)
    print(f"   {len(all_titles[cat])} articles")

→ Categoría:Deporte en Argentina


KeyboardInterrupt: 

In [51]:
# Union globale
union_titles = set().union(*all_titles.values())
union_norm = {normalize(t) for t in union_titles}

import pandas as pd 
df = pd.read_csv("/home/ppoulenard/LLM_BIAS_UCHILE/DATA/cultural_wiki_mcq_es_clean.csv")
print(df["title"])
df["title_norm"] = df["title"].astype(str).apply(normalize)


print(union_titles)
print(union_norm)
df_corpus = df[df["title_norm"].isin(union_norm)].copy()
print(f"\n✅ Corpus final : {len(df_corpus)} articles")
df_corpus.to_csv("/home/ppoulenard/LLM_BIAS_UCHILE/DATA/SUBSETS/artesania_articles_es.csv", index=False)

0        Museo Nacional del Virreinato
1           La Universidad Desconocida
2                         Tres (libro)
3                        Ángel Boligán
4                     Taller (revista)
                     ...              
18820                La Tribuna (Perú)
18821                        Yatoch_Ku
18822                    Gilberto Owen
18823          Guanajuato (Guanajuato)
18824                   Tafí del Valle
Name: title, Length: 18825, dtype: str
{'Artesanía y arte popular del Estado de México', 'Tecla Tofano', 'Artesanía en crin de Rari', 'Loceras de Pilén', 'Artesanías y arte popular en la Ciudad de México', 'Joyería de Bolivia', 'Museos El Ceibo', 'Metalurgia tradicional en México', 'Cristo de la Salud (Los Llanos de Aridane)', 'Rostros de bebé', 'Carlomagno Pedro Martínez', 'Celso Camacho Quiroz', 'Cerámica Tohil Plomiza', 'Andrés Uc Dzul', 'Rehundido de la plata', 'Carlos Seoane', 'Imelda Guillermina Barnett Díaz', 'Artesanías y arte folklórico mexicano', 'Cabinet

## RENDRE LES DATASETS DISJOINTS (POUR GNN et tout) :


In [56]:
import glob
import os
import pandas as pd

# Répertoire des datasets
DATA_DIR = "/home/ppoulenard/LLM_BIAS_UCHILE/DATA/SUBSETS"
PATTERN = os.path.join(DATA_DIR, "*_articles_es.csv")

# 1. Détection automatique des fichiers
files = sorted(glob.glob(PATTERN))
files = [f for f in files if not f.endswith("_disjoint.csv")]

print(f"{len(files)} fichiers détectés\n")

# 2. Chargement + déduplication interne sur title_norm
datasets = {}          # path -> DataFrame dédupliqué
raw_sizes = {}         # path -> taille brute (avant dédup interne)
orig_sizes = {}        # path -> taille après dédup interne
internal_dups = {}     # path -> nb de doublons internes supprimés

for f in files:
    df = pd.read_csv(f, sep=",", encoding="utf-8")
    if "title_norm" not in df.columns:
        raise ValueError(f"Colonne 'title_norm' absente dans {f}")
    raw_sizes[f] = len(df)
    df = df.drop_duplicates(subset="title_norm", keep="first").reset_index(drop=True)
    datasets[f] = df
    orig_sizes[f] = len(df)
    internal_dups[f] = raw_sizes[f] - orig_sizes[f]

# 3. Attribution de chaque title_norm au plus petit dataset
def sort_key(path):
    return (orig_sizes[path], os.path.basename(path))

winner = {}
for f in files:
    for tn in datasets[f]["title_norm"]:
        if tn not in winner or sort_key(f) < sort_key(winner[tn]):
            winner[tn] = f

# 4. Filtrage + écriture + collecte des stats
stats = []
final_dfs = {}
for f in files:
    df = datasets[f]
    mask = df["title_norm"].map(lambda tn: winner[tn] == f)
    df_out = df[mask].reset_index(drop=True)
    final_dfs[f] = df_out

    base, ext = os.path.splitext(f)
    out_path = f"{base}_disjoint{ext}"
    df_out.to_csv(out_path, sep=",", encoding="utf-8", index=False)

    cat = os.path.basename(f).replace("_articles_es.csv", "")
    n_raw = raw_sizes[f]
    n_dedup = orig_sizes[f]
    n_final = len(df_out)
    stats.append({
        "categorie": cat,
        "brut": n_raw,
        "doublons_internes": internal_dups[f],
        "apres_dedup": n_dedup,
        "supprimes_chevauchement": n_dedup - n_final,
        "final": n_final,
        "pct_supprime": round(100 * (n_dedup - n_final) / n_dedup, 2) if n_dedup else 0.0,
    })

stats_df = pd.DataFrame(stats).sort_values("brut", ascending=False).reset_index(drop=True)

# 5. Affichage du tableau par dataset
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
print("=" * 100)
print("STATISTIQUES PAR DATASET")
print("=" * 100)
print(stats_df.to_string(index=False))

# 6. Totaux globaux
total_raw = stats_df["brut"].sum()
total_internal = stats_df["doublons_internes"].sum()
total_dedup = stats_df["apres_dedup"].sum()
total_removed = stats_df["supprimes_chevauchement"].sum()
total_final = stats_df["final"].sum()

# nombre de title_norm uniques au global (= nb de lignes finales attendu)
unique_titles = len(winner)

print("\n" + "=" * 100)
print("STATISTIQUES GLOBALES")
print("=" * 100)
print(f"Nombre de datasets                          : {len(files)}")
print(f"Articles bruts (avant tout traitement)      : {total_raw}")
print(f"Doublons internes supprimés                 : {total_internal}")
print(f"Articles après dédup interne                : {total_dedup}")
print(f"Articles supprimés (chevauchement inter-ds) : {total_removed}")
print(f"Articles finaux (total)                     : {total_final}")
print(f"Titres uniques au global (contrôle)         : {unique_titles}")
print(f"  -> cohérent : {total_final == unique_titles}")
print(f"Réduction totale vs brut                    : "
      f"{round(100 * (total_raw - total_final) / total_raw, 2)}%")

# 7. Contrôle de disjonction entre fichiers de sortie
print("\n" + "=" * 100)
print("CONTRÔLE DE DISJONCTION")
print("=" * 100)
seen = {}
overlap_found = False
for f in files:
    cat = os.path.basename(f).replace("_articles_es.csv", "")
    for tn in final_dfs[f]["title_norm"]:
        if tn in seen:
            overlap_found = True
            print(f"  CHEVAUCHEMENT : '{tn}' dans {seen[tn]} et {cat}")
        else:
            seen[tn] = cat
if not overlap_found:
    print("  OK : aucun title_norm présent dans plusieurs fichiers de sortie.")

# 8. Sauvegarde du rapport de stats en CSV
report_path = os.path.join(DATA_DIR, "disjoint_stats_report.csv")
stats_df.to_csv(report_path, sep=",", encoding="utf-8", index=False)
print(f"\nRapport de stats sauvegardé : {report_path}")
print("\nTerminé.")



10 fichiers détectés

STATISTIQUES PAR DATASET
   categorie  brut  doublons_internes  apres_dedup  supprimes_chevauchement  final  pct_supprime
arquitectura  2036                152         1884                      231   1653         12.26
  literatura  1970                 58         1912                      397   1515         20.76
      musica  1943                 70         1873                      567   1306         30.27
        cine  1432                 58         1374                      105   1269          7.64
    folclore  1098                 37         1061                      486    575         45.81
    religion  1050                 81          969                       33    936          3.41
 gastronomia   403                  9          394                        6    388          1.52
       danza   399                 11          388                       10    378          2.58
     pintura   288                  2          286                        9    2

In [57]:
import glob
import os
import pandas as pd

DATA_DIR = "/home/ppoulenard/LLM_BIAS_UCHILE/DATA/SUBSETS"

# On travaille sur les fichiers disjoints générés précédemment
disjoint_files = sorted(glob.glob(os.path.join(DATA_DIR, "*_articles_es_disjoint.csv")))

print(f"{len(disjoint_files)} fichiers disjoints détectés\n")

char_stats = []
for f in disjoint_files:
    df = pd.read_csv(f, sep=",", encoding="utf-8")
    cat = os.path.basename(f).replace("_articles_es_disjoint.csv", "")

    if "content" not in df.columns:
        raise ValueError(f"Colonne 'content' absente dans {f}")

    # longueur en caractères de chaque cellule (NaN -> 0)
    lengths = df["content"].fillna("").astype(str).str.len()

    char_stats.append({
        "categorie": cat,
        "n_lignes": len(df),
        "total_chars": int(lengths.sum()),
        "moyenne_chars": round(lengths.mean(), 1) if len(df) else 0,
        "median_chars": int(lengths.median()) if len(df) else 0,
        "min_chars": int(lengths.min()) if len(df) else 0,
        "max_chars": int(lengths.max()) if len(df) else 0,
    })

char_df = pd.DataFrame(char_stats).sort_values("total_chars", ascending=False).reset_index(drop=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
print("=" * 100)
print("TAILLE EN CARACTÈRES DE LA COLONNE 'content' PAR DATASET (fichiers disjoints)")
print("=" * 100)
print(char_df.to_string(index=False))

# Total global
total_chars_all = char_df["total_chars"].sum()
total_lines_all = char_df["n_lignes"].sum()
print("\n" + "=" * 100)
print(f"TOTAL global de caractères (content) : {total_chars_all:,}".replace(",", " "))
print(f"TOTAL global de lignes               : {total_lines_all:,}".replace(",", " "))
print(f"Moyenne globale chars/ligne          : "
      f"{round(total_chars_all / total_lines_all, 1) if total_lines_all else 0}")

# Sauvegarde du rapport
report_path = os.path.join(DATA_DIR, "content_char_stats.csv")
char_df.to_csv(report_path, sep=",", encoding="utf-8", index=False)
print(f"\nRapport sauvegardé : {report_path}")


10 fichiers disjoints détectés

TAILLE EN CARACTÈRES DE LA COLONNE 'content' PAR DATASET (fichiers disjoints)
   categorie  n_lignes  total_chars  moyenne_chars  median_chars  min_chars  max_chars
      musica      1306     14518241        11116.6          6844        606     206558
arquitectura      1653     14064663         8508.6          5420        665     205366
  literatura      1515     12747862         8414.4          5557        644     128255
        cine      1269     12026748         9477.3          5654        786     162639
    religion       936     10454610        11169.5          6703        754     207125
    folclore       575      5386341         9367.5          5502        649      88400
 gastronomia       388      3314989         8543.8          5107        670     105447
       danza       378      2886464         7636.1          4864        508     144052
     pintura       277      2121144         7657.6          5317       1125      63556
   artesania       1